# E15 — Human Review Routing / Selective Prediction
Stage A + fresh DEV validation. Zero policy tuning; one validation run; no TEST.

In [1]:
import json

## 1. Research question

## 2. Stage A development dataset

In [2]:
a=json.load(open('results/routing_policy_eval_e13_full.json'));print(a['n'],a['joint_success'])

150 120


## 3. Observable routing signals

In [3]:
print(a['signals']['available']);print(a['signals']['not_available'])

['context_chars', 'error', 'evidence_chars', 'label', 'label_evidence_consistent', 'latency_ms', 'n_duplicate_quotes', 'n_hallucinated_v2', 'n_quotes', 'output_tokens', 'parse_status', 'retries', 'rule_label', 'unusable', 'input_tokens', 'retry_count', 'cost_usd']
['confidence/probability (GPT-P0 schema is {label,evidence} only)', 'logprobs (not requested/recorded)', 'retrieval scores (FULL context has no retrieval)', 'timeouts/provider errors: fields exist (error_type, retry_count) but are all None/0 in E13 FULL (150/150 strict parse, 0 errors, 0 retries)']


## 4. Policy definitions

## 5. Stage A routing results

In [4]:
for p,v in a['policies'].items(): print(p,{k:v['joint'][k] for k in ('review_rate','error_capture','residual_error_rate')})

R0 {'review_rate': 0.0, 'error_capture': 0.0, 'residual_error_rate': 0.2}
R1 {'review_rate': 0.02666666666666667, 'error_capture': 0.06666666666666667, 'residual_error_rate': 0.1917808219178082}
R2 {'review_rate': 0.3466666666666667, 'error_capture': 0.4, 'residual_error_rate': 0.1836734693877551}
R3 {'review_rate': 0.5733333333333334, 'error_capture': 0.9, 'residual_error_rate': 0.046875}


## 6. Provisional operational constraints

In [5]:
print(open('frozen_protocol.yaml').read()[:2500])

# FROZEN before the fresh-DEV model calls. Not to be modified after results are seen.
manifest: DEV_ROUTING_v1 (seed 1500, n=138: 60 E / 18 C / 60 NM; all remaining fresh Contradictions)
model: openai/gpt-5-mini via openrouter; prompt GPT-P0 (sha1 3fcc7c95...); FULL NDA context; temperature 0.0; concurrency 5; timeout 60s; run once, no reruns; run_id e15_dev_routing_validation
policies: {R0: none, R1: "source/structural integrity", R2: "R1 + all predicted Contradiction (reference only)", R3: "R1 + keyword-rule disagreement (primary candidate)"}  # implemented in scripts/e15_routing_stage_a.py, unmodified
provisional_constraints:   # provisional project-level values inherited from the earlier abstention plan; NOT measured capacity, SLA, or customer requirement
  MAX_DEV_STANDARDIZED_REVIEW_RATE: 0.40
  MAX_RESIDUAL_JOINT_ERROR_RATE: 0.10   # final-architecture operational proxy, not a validated legal-risk threshold
  standardization_weights: {Entailment: 519, Contradiction: 95, NotMenti

## 7. Fresh DEV manifest

In [6]:
print(json.load(open('results/manifest_verification.json')))

{'checks': {'seed_1500': True, 'n_138': True, 'class_counts_60_18_60': True, 'all_remaining_fresh_contradictions_used': True, 'disjoint_seed42_sample': True, 'disjoint_golden': True, 'disjoint_E13_DEV_ARCH_v1': True, 'gold_labels_match_dev': True}, 'manifest_sha256': 'aa2ef2c274c51266e6daba7b33d167b7f9215929d7a6081d9ba76cb8e9065855', 'n_docs': 41}


## 8. Execution integrity

In [7]:
print(json.load(open('results/pre_run_verification.json'))['checks']);print(json.load(open('results/run_E15_wall_seconds.json')))

{'manifest_frozen_138': True, 'full_ids_same_order': True, 'full_context_is_complete_dev_doc': True, 'no_truth_in_model_facing': True, 'prompt_is_gpt_p0': True, 'ledger_approx_2_8044': True, 'no_existing_e15_outputs': True, 'no_existing_e15_ledger_rows': True, 'budget_gate_pass': True}
{'spend_usd': 0.30771675000000004, 'wall_seconds': 230.33479229197837, 'stopped': None, 'successful_calls': 137, 'failed_calls': 1, 'verify': {'records_138': True, 'no_duplicates': True, 'all_ids': True, 'all_successful': False}, 'pre_run_ledger_usd': 2.80439895, 'post_run_ledger_usd': 3.112115699999999, 'completed': False}


## 9. Fresh model performance

In [8]:
v=json.load(open('results/validation_results.json'));print(v['model_performance'])

{'n': 138, 'accuracy': 0.7391304347826086, 'macro_f1': 0.72082067074096, 'joint': 0.7101449275362319, 'joint_n': 98, 'recall': {'Entailment': 0.85, 'Contradiction': 0.7777777777777778, 'NotMentioned': 0.6166666666666667}, 'contradiction_recall_ci95': [0.5478, 0.91], 'joint_by_class': {'Entailment': 0.7833333333333333, 'Contradiction': 0.7777777777777778, 'NotMentioned': 0.6166666666666667}, 'confusion': {'Entailment': {'Entailment': 51, 'Contradiction': 4, 'NotMentioned': 4, 'null': 1}, 'Contradiction': {'Contradiction': 14, 'Entailment': 4}, 'NotMentioned': {'NotMentioned': 37, 'Entailment': 16, 'Contradiction': 7}}, 'parse': {'strict': 137, 'invalid': 1}, 'cost_usd': 0.30771675, 'n_hallucinated_quote_cases': 5, 'source_valid_quote_rate': 0.97}


## 10. R0 results

In [9]:
P=lambda p:print(p,{k:v['policies'][p]['joint'][k] for k in ('review','review_rate','error_capture','residual_error_rate','selective_success','false_review_rate','routing_precision','confusion')},v['policies'][p]['dev_standardized_review_rate'])
P('R0')

R0 {'review': 0, 'review_rate': 0.0, 'error_capture': 0.0, 'residual_error_rate': 0.2898550724637681, 'selective_success': 0.7101449275362319, 'false_review_rate': 0.0, 'routing_precision': None, 'confusion': {'failure_review': 0, 'failure_auto': 40, 'success_review': 0, 'success_auto': 98}} 0.0


## 11. R1 results

In [10]:
P('R1')

R1 {'review': 6, 'review_rate': 0.043478260869565216, 'error_capture': 0.125, 'residual_error_rate': 0.26515151515151514, 'selective_success': 0.7348484848484849, 'false_review_rate': 0.01020408163265306, 'routing_precision': 0.8333333333333334, 'confusion': {'failure_review': 5, 'failure_auto': 35, 'success_review': 1, 'success_auto': 97}} 0.04525340190721097


## 12. R2 results

In [11]:
P('R2')

R2 {'review': 29, 'review_rate': 0.21014492753623187, 'error_capture': 0.375, 'residual_error_rate': 0.22935779816513763, 'selective_success': 0.7706422018348624, 'false_review_rate': 0.14285714285714285, 'routing_precision': 0.5172413793103449, 'confusion': {'failure_review': 15, 'failure_auto': 25, 'success_review': 14, 'success_auto': 84}} 0.18557269902496518


## 13. R3 results

In [12]:
P('R3')

R3 {'review': 71, 'review_rate': 0.5144927536231884, 'error_capture': 0.825, 'residual_error_rate': 0.1044776119402985, 'selective_success': 0.8955223880597015, 'false_review_rate': 0.3877551020408163, 'routing_precision': 0.4647887323943662, 'confusion': {'failure_review': 33, 'failure_auto': 7, 'success_review': 38, 'success_auto': 60}} 0.5127665273759776


## 14. DEV-standardized workload estimate

In [13]:
print({p:(v['policies'][p]['workload_raw'],v['policies'][p]['workload_dev_standardized']) for p in v['policies']})

{'R0': ({'100': 0.0, '1000': 0.0, '8000': 0.0}, {'100': 0.0, '1000': 0.0, '8000': 0.0}), 'R1': ({'100': 4.3, '1000': 43.5, '8000': 347.8}, {'100': 4.5, '1000': 45.3, '8000': 362.0}), 'R2': ({'100': 21.0, '1000': 210.1, '8000': 1681.2}, {'100': 18.6, '1000': 185.6, '8000': 1484.6}), 'R3': ({'100': 51.4, '1000': 514.5, '8000': 4115.9}, {'100': 51.3, '1000': 512.8, '8000': 4102.1})}


## 15. Contradiction analysis

In [14]:
print(v['contradiction_descriptive_n18'])

{'R0': {'joint_failures': 4, 'failures_caught': 0, 'residual_failures': 4, 'correct_reviewed': 0, 'correct_total': 14}, 'R1': {'joint_failures': 4, 'failures_caught': 0, 'residual_failures': 4, 'correct_reviewed': 1, 'correct_total': 14}, 'R2': {'joint_failures': 4, 'failures_caught': 0, 'residual_failures': 4, 'correct_reviewed': 14, 'correct_total': 14}, 'R3': {'joint_failures': 4, 'failures_caught': 3, 'residual_failures': 1, 'correct_reviewed': 11, 'correct_total': 14}}


## 16. Stage A vs validation generalization

In [15]:
print(v['stageA_vs_validation_R3'])

{'review_rate': {'stageA': 0.5733333333333334, 'validation': 0.5144927536231884}, 'error_capture': {'stageA': 0.9, 'validation': 0.825}, 'residual_error_rate': {'stageA': 0.046875, 'validation': 0.1044776119402985}, 'routing_precision': {'stageA': 0.313953488372093, 'validation': 0.4647887323943662}, 'false_review_rate': {'stageA': 0.49166666666666664, 'validation': 0.3877551020408163}}


## 17. Residual failures

In [16]:
print(v['residual_buckets_R3']);[print(x['case_id'],x['gold'],x['pred'],x['bucket']) for x in v['residual_failures_R3']]

{'missed_contradiction': 1, 'reasoning_error_entailment': 4, 'over_inference': 2}
dev::586::nda-4 Contradiction Entailment missed_contradiction
dev::72::nda-16 Entailment NotMentioned reasoning_error_entailment
dev::381::nda-1 Entailment NotMentioned reasoning_error_entailment
dev::411::nda-4 Entailment NotMentioned reasoning_error_entailment
dev::605::nda-10 Entailment NotMentioned reasoning_error_entailment
dev::37::nda-10 NotMentioned Entailment over_inference
dev::590::nda-16 NotMentioned Entailment over_inference


[None, None, None, None, None, None, None]

## 18. Workload/safety decision

In [17]:
print(v['selection'])

{'criteria': {'1_std_review<=40%': False, '2_raw_residual<10%': False, '2b_std_residual<10% (info)': False, '3_capture_gap_vs_R1>=20pp': True, '4_no_C_safety_regression': True, '5_no_structural_problem': True}, 'R3_selected': False, 'R1_useful_structural_guard': 0.125, 'R1_capture': 0.125}


## 19. Final A/B/C outcome

In [18]:
print('Outcome C: no effective selective routing; R1 retained only as structural source-integrity safeguard')

Outcome C: no effective selective routing; R1 retained only as structural source-integrity safeguard


## 20. Limitations

In [19]:
print(open('summary.md').read().split('## Limitations')[-1])


n=138 (18 C; Contradiction descriptive); DEV with disclosed historical exposure; standardization assumes validation within-class mix ≈ DEV; provisional constraints not validated; model performance dropped vs E13 (variance/difficulty); one unusable parse; residual 10.4% vs 10% is marginal and CI is wide (5.2–20.0%) — the workload failure (51% vs 40%) is the clearer reason for outcome C.

